# Clase 015 — NumPy: ufuncs y vectorización

**Parte 0** · VanderPlas cap. 2 § 2.3.

> 🎯 Abandonar `for` sobre arrays. Ufuncs = funciones C vectorizadas elementwise — el speedup real.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import numpy as np
import time, tracemalloc
rng = np.random.default_rng(42)

## 1️⃣ ¿Qué es una ufunc?

Una **universal function** es una función NumPy que opera **elementwise** sobre arrays, implementada en C y vectorizada (SIMD cuando posible).

**Unarias** (un input): `np.exp`, `np.log`, `np.sin`, `np.sqrt`, `np.abs`, `np.negative`...
**Binarias** (dos inputs): `np.add`, `np.multiply`, `np.divide`, `np.power`, `np.maximum`...

Los **operadores** (`+`, `-`, `*`, `/`, `**`, `==`, `<`...) son sintaxis dulce sobre ufuncs.

## 2️⃣ El speedup en vivo

In [ ]:
N = 1_000_000
lst = list(range(N))
arr = np.arange(N)

# Versión Python
t0 = time.perf_counter()
res_py = [x*x + 2*x + 1 for x in lst]
t1 = time.perf_counter()

# Versión vectorizada
t2 = time.perf_counter()
res_np = arr*arr + 2*arr + 1
t3 = time.perf_counter()

print(f'Python loop : {(t1-t0)*1000:.1f} ms')
print(f'NumPy vec   : {(t3-t2)*1000:.1f} ms')
print(f'speedup     : {(t1-t0)/(t3-t2):.0f}×')

## 3️⃣ Operadores como ufuncs

```python
a + b   ≡  np.add(a, b)
a * b   ≡  np.multiply(a, b)
a ** b  ≡  np.power(a, b)
a == b  ≡  np.equal(a, b)
a > b   ≡  np.greater(a, b)
-a      ≡  np.negative(a)
```

Esto significa que `arr*arr + 2*arr + 1` son 4 ufuncs encadenadas — cada una alloca un array temporal. Para ahorrar memoria, usa `out=`:

In [ ]:
# Sin out=: cada operación alloca
A = rng.random(1_000_000)
tracemalloc.start()
result = A * A + 2*A + 1
_, peak1 = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'sin out= : peak {peak1/1024:.0f} KB')

# Con out=: in-place, sin allocs
A = rng.random(1_000_000)
tracemalloc.start()
np.multiply(A, A, out=A)
np.multiply(2, A, out=A)   # nota: el segundo factor podría ser otro array
np.add(A, 1, out=A)
_, peak2 = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'con out= : peak {peak2/1024:.0f} KB')
print(f'ratio    : {peak1/max(peak2,1):.1f}×')

## 4️⃣ Ufuncs trigonométricas, exponenciales y logarítmicas

VanderPlas tabla 2-4:

In [ ]:
x = np.linspace(0, 2*np.pi, 5)
print('x       :', x)
print('sin(x)  :', np.sin(x))
print('cos(x)  :', np.cos(x))
print()
print('exp(x)  :', np.exp(x[:3]))
print('log(...):', np.log(np.exp(x[:3])))   # log(exp(x)) ≈ x
print('sqrt    :', np.sqrt([1, 4, 9, 16]))
print('abs     :', np.abs([-3, 5, -7]))

## 5️⃣ `np.where` — ternario vectorizado

```python
np.where(cond_array, valor_si_true, valor_si_false)
```

Util para clasificar, máscaras, sustituciones:

In [ ]:
notas = np.array([2.8, 4.5, 6.1, 3.2, 7.0, 5.5])
estado = np.where(notas >= 4, 'aprobado', 'reprobado')
for n, e in zip(notas, estado):
    print(f'{n}: {e}')

## 6️⃣ ⚠️ Trampas

**Overflow silencioso** (ya visto en clase 014). NumPy no para, sólo wrap-around.

**NaN propagación**: cualquier operación con NaN produce NaN:

```python
np.array([1, 2, np.nan, 4]).sum()    # nan
np.array([1, 2, np.nan, 4]).mean()   # nan
```

**Fix**: usa las variantes `nan*`:

```python
np.nansum(arr)     # ignora NaN
np.nanmean(arr)    # ignora NaN
np.nanmedian(arr)
```

**División por cero**: produce `inf` con warning. Para silenciar (o convertir a NaN), usa `np.errstate`:

In [ ]:
a = np.array([1, 2, np.nan, 4, 5])
print(f'sum      : {a.sum()}')           # nan
print(f'nansum   : {np.nansum(a)}')      # 12
print(f'mean     : {a.mean()}')
print(f'nanmean  : {np.nanmean(a)}')

print()
with np.errstate(divide='ignore', invalid='ignore'):
    res = np.array([1, 0, -1]) / np.array([0, 0, 0])
    print('1/0, 0/0, -1/0 :', res)   # inf, nan, -inf

## ✅ Checklist

- [ ] Sé qué es una ufunc y por qué es rápida
- [ ] Reescribo `for+append` como expresión vectorizada
- [ ] Uso `out=` para ahorrar memoria
- [ ] Conozco `np.where` para ternarios vectorizados
- [ ] Manejo NaN con `nansum`/`nanmean`

## 📝 Homework

Ver `README.md`. 3 loops reescritos con benchmark, demo `out=`, `np.where`, manejo NaN.

## 📖 Definiciones y características

**Ufunc (universal function)**

Función NumPy implementada en C que opera **elementwise** y vectorizada (SIMD cuando posible). Características: rápida (10-100× vs Python), broadcasting automático, soporta `out=` para in-place.

**Vectorización**

Operar sobre arrays completos en vez de loops Python: `arr * 2` en vez de `[x*2 for x in arr]`. La operación corre en C compilado sobre memoria contigua, sin overhead del intérprete por elemento.

**In-place (`out=`)**

Escribir el resultado de una ufunc en un array existente, sin allocar memoria nueva: `np.multiply(a, 2, out=a)`. Útil con arrays grandes donde la copia temporal duplicaría la memoria pico.

**Propagación de NaN**

Cualquier operación que tenga `NaN` como input produce `NaN`. `np.array([1, np.nan, 3]).sum()` → `nan`. Para ignorar usa variantes `nan*`: `nansum`, `nanmean`, `nanmedian`.

**`np.where(cond, a, b)`**

Ternario vectorizado: para cada elemento, si `cond` es True usa `a`, si no usa `b`. Equivale a `[a if c else b for c, a, b in zip(cond, a, b)]` pero ~100× más rápido.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `for i in range(len(arr)): arr[i] = ...` es lentísimo | Loops Python sobre array NumPy = lo peor de ambos mundos. **Fix**: reescribe como expresión vectorizada (`arr = ...`) o usa ufunc explícita. |
| RuntimeWarning: divide by zero / invalid value | NumPy avisa pero no para: `1/0` → `inf`, `0/0` → `nan`. **Fix**: filtra antes (`arr[arr != 0]`) o silencia con `np.errstate(divide='ignore', invalid='ignore')`. |
| `out=` con dtype incompatible | `np.add(int_arr, 0.5, out=int_arr)` falla — float no cabe en int. **Fix**: convierte primero (`arr = arr.astype(float)`) o usa array distinto como destino. |
| Resultado de `np.where` no es lo esperado | Los 3 args se evalúan **completos**: `np.where(arr>0, 1/arr, 0)` calcula `1/arr` para TODOS los elementos (incluso negativos → division por cero). **Fix**: usa `np.where` solo para values planos, no expresiones. |
| `arr.sum()` da NaN y no sé por qué | Hay un NaN escondido en el array. **Fix**: `print(np.isnan(arr).sum())` para contar; usa `np.nansum()` para ignorar. |

## ❓ Preguntas frecuentes

**❓ ¿Cuánto más rápido es vectorizar?**

Típicamente 50-100× para arrays de 1M elementos. Para arrays pequeños (<100), la ganancia es menor o nula (overhead constante). Mide con `%timeit`, no asumas.

**❓ ¿`arr + 1` o `np.add(arr, 1)`?**

Equivalentes. Operadores son sintaxis dulce sobre ufuncs. Usa `np.add(...)` cuando necesitas `out=` (in-place) o `where=` (mask).

**❓ ¿NumPy aprovecha mi GPU?**

**No** — solo CPU. Para GPU: CuPy (drop-in replacement), PyTorch tensors, JAX. NumPy 2 está mejorando vectorización CPU (SIMD wider, BLAS) pero sigue siendo CPU.

**❓ ¿Por qué `arr ** 2` es más rápido que `arr * arr`?**

Suelen empatar (`**` también es ufunc). Para potencias enteras pequeñas (2, 3), NumPy a veces usa atajos. Mide con `%timeit` en tu caso específico.

**❓ ¿`np.where` o boolean mask?**

**Mask** (`arr[cond] = valor`) si vas a modificar in-place o filtrar (`arr[arr>0]`). **`np.where(cond, a, b)`** si necesitas un array nuevo con dos valores posibles según condición.

## 🔗 Referencias

- VanderPlas cap. 2 § 2.3
- [ufuncs reference](https://numpy.org/doc/stable/reference/ufuncs.html)

➡️ **Siguiente:** [016 — Agregaciones](../016-numpy-agregaciones/README.md)

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero; acá está una solución de referencia comentada. Cada celda es autocontenida (re-importa lo que usa) y verifica el resultado con `assert`/`print`.

**Ejercicio 1.** Compara `[x*x + 2*x + 1 for x in range(1_000_000)]` vs `arr*arr + 2*arr + 1` y mide el tiempo.

In [ ]:
# Ejercicio 1 - Benchmark: loop Python vs vectorizado
import time
import numpy as np

N = 1_000_000
arr = np.arange(N)

t0 = time.perf_counter()
python_res = [x*x + 2*x + 1 for x in range(N)]
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
np_res = arr*arr + 2*arr + 1
t_vec = time.perf_counter() - t0

speedup = t_loop / t_vec
print(f"loop Python : {t_loop*1000:.1f} ms")
print(f"vectorizado : {t_vec*1000:.1f} ms")
print(f"speedup     : {speedup:.0f}x")

# El resultado numerico es identico; el vectorizado es mucho mas rapido.
assert np.array_equal(np.array(python_res), np_res)
assert speedup > 1
print("OK: mismo resultado, la version vectorizada es mas rapida")


**Ejercicio 2.** Con `np.exp` y `np.log`, verifica que `log(exp(x)) ≈ x` para 1000 valores y reporta el error máximo.

In [ ]:
# Ejercicio 2 - Logaritmo y exponencial son inversas
import numpy as np

rng = np.random.default_rng(0)
x = rng.uniform(-5, 5, 1000)

recuperado = np.log(np.exp(x))       # deberia devolver x
error_max = np.max(np.abs(recuperado - x))
print(f"error maximo = {error_max:.2e}")

# El error es de nivel de redondeo de punto flotante (~1e-15).
assert np.allclose(recuperado, x)
assert error_max < 1e-9
print("OK: log(exp(x)) == x salvo error de redondeo")


**Ejercicio 3.** Compara `arr = arr*2 + 1` vs `np.multiply(arr, 2, out=arr); np.add(arr, 1, out=arr)` con `tracemalloc`.

In [ ]:
# Ejercicio 3 - In-place (out=) vs allocacion nueva
import tracemalloc
import numpy as np

N = 1_000_000

# --- Version que ALLOCA arrays temporales ---
arr_a = np.arange(N, dtype=np.float64)
tracemalloc.start()
arr_a = arr_a * 2 + 1
pico_alloc = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

# --- Version IN-PLACE con out= (no crea arrays nuevos) ---
arr_b = np.arange(N, dtype=np.float64)
tracemalloc.start()
np.multiply(arr_b, 2, out=arr_b)
np.add(arr_b, 1, out=arr_b)
pico_inplace = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f"pico con alloc   : {pico_alloc/1e6:.2f} MB")
print(f"pico in-place    : {pico_inplace/1e6:.2f} MB")

# Mismo resultado numerico; in-place usa mucha menos memoria extra.
assert np.array_equal(arr_a, arr_b)
assert pico_inplace < pico_alloc
print("OK: out= evita los arrays temporales -> menor memoria pico")


**Ejercicio 4.** Dado un array de notas, crea otro con `'aprobado'` si nota >= 4 y `'reprobado'` si no, usando `np.where`.

In [ ]:
# Ejercicio 4 - np.where como ternario vectorizado
import numpy as np

notas = np.array([2.8, 4.5, 6.1, 3.2, 7.0, 5.5, 4.0])
estado = np.where(notas >= 4, "aprobado", "reprobado")
print(list(zip(notas.tolist(), estado.tolist())))

assert estado[0] == "reprobado" and estado[1] == "aprobado"
assert estado[6] == "aprobado"     # 4.0 cumple el umbral >= 4
assert (estado == "aprobado").sum() == 5
print("OK: np.where clasifico las notas sin loop")


**Ejercicio 5.** Calcula `.sum()` y `.mean()` de un array con NaN y compáralos con `np.nansum` y `np.nanmean`.

In [ ]:
# Ejercicio 5 - Trampa NaN: propagacion vs variantes nan*
import numpy as np

a = np.array([1, 2, np.nan, 4])

s, m = a.sum(), a.mean()                    # cualquier NaN contamina el total
ns, nm = np.nansum(a), np.nanmean(a)        # las variantes nan* lo ignoran
print(f"sum={s} mean={m}  |  nansum={ns} nanmean={nm}")

assert np.isnan(s) and np.isnan(m)          # se propaga el NaN
assert ns == 7.0                            # 1+2+4
assert nm == 7.0 / 3                        # promedio de los 3 valores validos
print("OK: sum/mean propagan NaN; nansum/nanmean lo ignoran")
